# 04 — Model: XGBoost

Requires `train_features.csv` / `test_features.csv` from **01_feature_engineering.ipynb**.

Run `!pip install xgboost` once if you don't already have it.

XGBoost needs categoricals either one-hot/label-encoded or passed via `enable_categorical=True` with the
`category` dtype (used here) and `tree_method='hist'`. Saves `oof_xgb.csv` and `test_pred_xgb.csv` for the
ensembling notebook.

In [1]:
!pip install -q xgboost


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

DATA_DIR = "."   # <-- folder with train_features.csv / test_features.csv from notebook 01
N_FOLDS = 5
SEED = 42

train_fe = pd.read_csv(f"{DATA_DIR}/train_features.csv")
test_fe = pd.read_csv(f"{DATA_DIR}/test_features.csv")

num_cols = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours',
            'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time',
            'notif_per_hour', 'app_per_hour', 'mins_per_appopen', 'mins_per_notif', 'productivity',
            'sleep_screen_sum', 'nonscreen_hours', 'screen_plus_weekend', 'screen_sleep_ratio',
            'sm_ratio', 'game_ratio', 'work_ratio', 'screen_minus_work', 'weekday_weekend_ratio',
            'screen_x_sm', 'screen_x_weekend', 'sm_x_weekend', 'screen_x_sleep']
cat_cols = ['gender', 'stress_level', 'academic_work_impact']
feat_cols = num_cols + cat_cols

X = train_fe[feat_cols].copy()
Xtest = test_fe[feat_cols].copy()
y = train_fe['addicted_label'].values

for c in cat_cols:
    X[c] = X[c].astype('category')
    Xtest[c] = Xtest[c].astype('category')

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))
print(X.shape, Xtest.shape)

import xgboost as xgb
import time


(691369, 30) (296302, 30)


In [3]:
oof_xgb = np.zeros(len(X))
test_xgb = np.zeros(len(Xtest))

xgb_params = dict(
    objective='binary:logistic', eval_metric='auc', learning_rate=0.03, max_depth=8,
    min_child_weight=10, subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0,
    n_estimators=3000, tree_method='hist', enable_categorical=True, random_state=SEED
)

t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(folds):
    model = xgb.XGBClassifier(**xgb_params, early_stopping_rounds=100)
    model.fit(
        X.iloc[tr_idx], y[tr_idx],
        eval_set=[(X.iloc[va_idx], y[va_idx])],
        verbose=False
    )
    p_va = model.predict_proba(X.iloc[va_idx])[:, 1]
    oof_xgb[va_idx] = p_va
    test_xgb += model.predict_proba(Xtest)[:, 1] / N_FOLDS
    print(f"fold {fold} auc={roc_auc_score(y[va_idx], p_va):.5f}  best_iter={model.best_iteration}  ({time.time()-t0:.0f}s elapsed)")

print("XGBoost OOF AUC:", roc_auc_score(y, oof_xgb))


fold 0 auc=0.96351  best_iter=2543  (196s elapsed)
fold 1 auc=0.96413  best_iter=2296  (373s elapsed)
fold 2 auc=0.96412  best_iter=1933  (522s elapsed)
fold 3 auc=0.96495  best_iter=2500  (710s elapsed)
fold 4 auc=0.96399  best_iter=2183  (876s elapsed)
XGBoost OOF AUC: 0.9641394102315409


In [4]:
pd.DataFrame({'id': train_fe['id'], 'oof_pred': oof_xgb}).to_csv(f"{DATA_DIR}/oof_xgb.csv", index=False)
pd.DataFrame({'id': test_fe['id'], 'test_pred': test_xgb}).to_csv(f"{DATA_DIR}/test_pred_xgb.csv", index=False)
print("saved oof_xgb.csv and test_pred_xgb.csv")


saved oof_xgb.csv and test_pred_xgb.csv
